# Archiver: Writing Model Predictions to Zarr

The `Archiver` class (`crest/archiver/Archiver.py`) handles writing batches of
model predictions to a zarr store on disk. It is built on two internal components:

- **StageWriter** — stages incoming coordinate/value pairs into parquet fragments,
  organized into hash buckets for efficient random-access writes.
- **ZarrWriter** — periodically consolidates staged parquet fragments into the
  final zarr array, supporting averaged writes when the same coordinate is written
  more than once.

This notebook covers:

1. **Data schema** — an `xr.DataArray` that defines coordinate dimensions and
   chunking for the output zarr.
2. **Archiving predictions** — opening an `Archiver`, supplying coordinate and
   feature arrays via `archive()`, and closing it to flush remaining data.
3. **Reading back** — loading and verifying the archived zarr with `xr.open_zarr`.

In [ ]:
# Allow crest to be imported from the examples/ sub-directory
%cd -q ..

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

from crest.archiver import Archiver

## 1. Data Schema

The `Archiver` requires an `xr.DataArray` (or `xr.Dataset`) as its `data_schema`
argument. The schema defines:

- **Coordinate dimensions** — axis names and tick values (e.g. `datetime`,
  `latitude`, `longitude`).
- **Chunking** — set via `.chunk(...)` on the `DataArray`; controls how zarr
  splits the array on disk. If the schema is unchunked, `'auto'` chunking is used.

When an `xr.Dataset` is passed, `Archiver` extracts the first variable as the
schema `DataArray`. The schema's *values* are ignored — only coordinates and
chunking matter.

Below we create a synthetic 3-D grid representing a daily dataset:
10 days × 5 latitudes × 4 longitudes.

In [ ]:
# Coordinate axes
times = pd.date_range('2020-01-01', periods=10, freq='D')
lats  = np.linspace(25.0, 45.0, 5)
lons  = np.linspace(-100.0, -80.0, 4)

# Schema DataArray — values are zeros; only coordinates and chunk layout are used
schema = xr.DataArray(
    np.zeros((len(times), len(lats), len(lons))),
    dims=['datetime', 'latitude', 'longitude'],
    coords={'datetime': times, 'latitude': lats, 'longitude': lons},
).chunk({'datetime': 10, 'latitude': 5, 'longitude': 4})

print(schema)

## 2. Archiving Predictions

`Archiver` is used as a context manager with two required arguments:

- `data_schema` — an `xr.DataArray` or `xr.Dataset` defining the coordinate grid.
- `output_path` — destination path for the zarr store.

Within the `with` block, call `archiver.archive(coords, values)` once per batch:

- `coords` — `dict[str, np.ndarray]` mapping each dimension name to a 1-D array
  of real coordinate values for that batch.
- `values` — `dict[str, np.ndarray]` mapping feature names to 1-D value arrays.

All arrays in `coords` and `values` must have the same length. Multiple calls
accumulate in parquet staging buffers; data is flushed to zarr every `write_every`
calls (default 1000) and automatically on context-manager exit.

Coordinate matching rules (set in `StageWriter.__init__`):
- **datetime** — uses *strict* matching; archive values must exactly equal a
  schema tick. Pass `times.values[i]` (a `datetime64[ns]` scalar) to guarantee
  an exact match.
- **latitude, longitude** — use *nearest-neighbor* matching by default.

Below we simulate archiving model predictions one time step at a time.

In [ ]:
OUTPUT = 'sm_predictions.zarr'

with Archiver(schema, OUTPUT, overwrite=True) as archiver:
    # Archive all spatial points for one time step per iteration.
    # In a real workflow this loop body would call model.predict_on_batch(x).
    for i, _ in enumerate(times):
        lat_grid, lon_grid = np.meshgrid(lats, lons, indexing='ij')

        coords = {
            'datetime' : np.full(lat_grid.size, times.values[i]),  # datetime64[ns]
            'latitude' : lat_grid.ravel(),
            'longitude': lon_grid.ravel(),
        }

        # Synthetic soil-moisture prediction (replace with model.predict_on_batch output)
        sm = np.sin(np.deg2rad(lat_grid.ravel())) * np.cos(np.deg2rad(lon_grid.ravel()))
        values = {'sm': sm}

        archiver.archive(coords, values)

print('Archiving complete.')

## 3. Reading Back Archived Data

The zarr store is self-describing: `xr.open_zarr` restores coordinate axes and
returns a lazy `xr.Dataset`. Pass `mask_and_scale=False` when the schema uses an
integer dtype to prevent NaN-fill casting.

We verify that the archived values exactly match the expected synthetic predictions.

In [ ]:
result = xr.open_zarr(OUTPUT)
print(result)

In [ ]:
result['sm']

In [ ]:
# Recompute expected values and compare against every archived time step
lat_grid, lon_grid = np.meshgrid(lats, lons, indexing='ij')
expected = np.sin(np.deg2rad(lat_grid)) * np.cos(np.deg2rad(lon_grid))

for t_idx in range(len(times)):
    actual = result['sm'].isel(datetime=t_idx).values
    np.testing.assert_allclose(actual, expected, rtol=1e-5,
                               err_msg=f'Mismatch at time index {t_idx}')

print(f'Verified {len(times)} time steps × {len(lats)} lats × {len(lons)} lons.')
print(f'Archived shape: {result.sizes}')

In [ ]:
# Clean up dir tree
import shutil
shutil.rmtree(OUTPUT)